© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

English translation managed by Tamás Takács. The translation was produced with AI assistance.

# **Hungarian AI Olympiad Online Qualifier, 2026/B - Production Line Quality Control**

This **Notebook** was created for the online round of the *2026 International AI Olympiad*, and provides a basic starting point for the contestants.

The notebook covers loading the data and its basic visualization, as well as a simple baseline model based on **logistic regression**. This model uses only the **labeled** training data.

During the competition you may use any package or framework, as long as the submitted solution complies with the rules stated on the **Kaggle competition page**.

Both the data provided for the competition and the task itself are **entirely synthetic**, so there is no point in looking for data from external sources.

**Important:** The training data has two parts, a **labeled** one (150 samples) and an **unlabeled** one (12,850 samples). It is worth using the unlabeled data during modeling as well!

# **0. Loading the Required Packages**

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler

# **1. Loading the Data**

During the competition four files are available to you:

- `train_labeled.csv`: Contains the **labeled** training data (150 samples). Each product has sensor and process features, as well as the target variable (`Selejt`), which indicates whether the product was defective (1 - defective, 0 - acceptable). Every row is identified by a unique `ID`.

- `train_unlabeled.csv`: Contains the **unlabeled** training data (12,850 samples). It has the same features, **but the `Selejt` target variable is missing**. You may use this data creatively during modeling.

- `test.csv`: The test data used for evaluating the competition, **without** the target variable (`Selejt`). Your goal is to produce predictions for this set.

Your task is to **train a machine learning model** on the training data that is able to estimate whether the products in the `test.csv` file are defective. The model must return a **probability value between 0 and 1** for every `ID` (1 - certainly defective, 0 - certainly acceptable).

## **1.1 Reading in the Data**

In [4]:
import os

# On Kaggle the data is available in the /kaggle/input/<competition-slug>/ folder.
# On Colab or locally we read from the current directory.
KAGGLE_DIR = '/kaggle/input/magyar-mi-diakolimpia-online-valogato-2026-b/'
INPUT_DIR = KAGGLE_DIR if os.path.exists(KAGGLE_DIR) else './'

train_labeled = pd.read_csv(INPUT_DIR + 'train_labeled.csv')
train_unlabeled = pd.read_csv(INPUT_DIR + 'train_unlabeled.csv')
test_df = pd.read_csv(INPUT_DIR + 'test.csv')

## **1.2 Overview of the Data**

In [5]:
print(f"Labeled training data: {len(train_labeled)} samples")
print(f"Unlabeled training data: {len(train_unlabeled)} samples")
print(f"Test data: {len(test_df)} samples")
print(f"\nNumber of columns: {len(train_labeled.columns)}")
print(f"Proportion of defective products (labeled): {train_labeled['Selejt'].mean():.1%}")

Labeled training data: 150 samples
Unlabeled training data: 12850 samples
Test data: 2000 samples

Number of columns: 35
Proportion of defective products (labeled): 45.3%


In [6]:
train_labeled.head()

,ID,Hőmérséklet,Nyomás,Rezgés,Feszültség,Nyomaték,Szerszám_Kopás,Karbantartás_Óta_Eltelt_Idő,Páratartalom,Fordulat,...,Szenzor_11,Szenzor_12,Szenzor_13,Szenzor_14,Szenzor_15,Szenzor_16,Szenzor_17,Szenzor_18,Gyártósor,Selejt
0,ITEM_00000,75.907041,3.831936,51.472834,217.098007,38.316036,228.361172,155.478734,64.597543,1389.297668,...,-0.742682,-0.902517,1.159349,-0.379631,1.747762,-1.598339,-2.352235,0.497678,Vonal_E,1
1,ITEM_00001,81.242746,5.357322,25.553280,230.298673,48.384111,21.935034,69.234019,35.211917,1672.997915,...,-0.750481,1.624672,0.744649,-1.176670,0.750373,-0.302829,0.939427,-0.567894,Vonal_A,0
2,ITEM_00002,74.004248,3.971429,19.762415,205.499735,32.403428,167.193096,124.985486,72.651122,1183.049348,...,-1.450498,1.675769,1.374522,-0.527704,-0.278072,0.208900,0.857715,0.147685,Vonal_B,1
3,ITEM_00003,73.061348,2.542578,44.395338,233.853194,31.420680,113.907234,103.992968,61.111509,1490.947877,...,-1.158815,0.822962,1.127353,-1.257186,1.012118,0.170600,0.387113,1.377332,Vonal_C,0
4,ITEM_00004,82.929130,4.704838,18.821581,237.629698,67.618695,66.443431,69.115467,40.095704,1790.750438,...,-0.853669,-0.341559,2.165415,-0.228110,0.275952,0.705621,-0.695425,2.051832,Vonal_B,1


## **1.3 Column Descriptions**

On the production line the following sensor and process features are recorded for every product:

**Physical sensor data:**
- `Hőmérséklet` *(float)*: The temperature of the manufacturing process (°C)
- `Nyomás` *(float)*: The manufacturing pressure (bar)
- `Rezgés` *(float)*: The level of machine vibration (mm/s)
- `Feszültség` *(float)*: The electrical voltage (V)
- `Nyomaték` *(float)*: The mechanical torque (Nm)
- `Páratartalom` *(float)*: The ambient humidity (%)
- `Fordulat` *(float)*: The rotational speed (RPM)

**Maintenance data:**
- `Szerszám_Kopás` *(float)*: The degree of tool wear (units)
- `Karbantartás_Óta_Eltelt_Idő` *(float)*: Time elapsed since the last maintenance (hours)

**Derived features:**
- `Hő_Nyomás_Szorzat` *(float)*: Product of temperature and pressure (scaled)
- `Rezgés_Nyomaték_Arány` *(float)*: Ratio of vibration to torque
- `Kopás_Karb_Szorzat` *(float)*: Product of wear and maintenance time (scaled)
- `Feszültség_Fordulat_Arány` *(float)*: Ratio of voltage to rotational speed (scaled)
- `Pára_Hő_Szorzat` *(float)*: Product of humidity and temperature (scaled)

**Sensor measurements:**
- `Szenzor_01` – `Szenzor_18` *(float)*: Other sensor values from the production line

**Other:**
- `Gyártósor` *(string)*: The identifier of the production line (Vonal_A, Vonal_B, Vonal_C, Vonal_D, Vonal_E)
- `Selejt` *(int)*: The **binary target variable**: 1 if the product is defective; 0 if it is acceptable. *(Present only in the labeled data.)*

In [7]:
train_labeled.describe()

,Hőmérséklet,Nyomás,Rezgés,Feszültség,Nyomaték,Szerszám_Kopás,Karbantartás_Óta_Eltelt_Idő,Páratartalom,Fordulat,Hő_Nyomás_Szorzat,...,Szenzor_10,Szenzor_11,Szenzor_12,Szenzor_13,Szenzor_14,Szenzor_15,Szenzor_16,Szenzor_17,Szenzor_18,Selejt
count,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,...,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000
mean,72.548247,3.953798,30.858938,225.060545,42.458176,131.512759,104.941162,56.900016,1466.098614,2.998981,...,0.000645,-0.103339,0.051693,-0.083692,-0.070300,0.266661,-0.007969,-0.139652,-0.079121,0.453333
std,11.022928,1.198175,11.844017,12.661402,11.460764,56.568590,44.944197,11.150658,277.456352,1.379008,...,1.048216,1.026141,1.024658,1.015823,0.949937,0.927979,0.947235,1.042249,1.014686,0.499485
min,44.525114,1.582984,7.734813,198.202370,15.833331,21.935034,12.991180,35.211917,777.032503,0.565813,...,-2.820573,-2.978052,-2.240823,-3.490971,-2.448686,-2.197009,-2.717297,-2.559200,-2.427527,0.000000
25%,64.280259,2.958320,21.242027,215.535315,34.128077,88.715653,69.111646,48.679227,1269.728738,2.097512,...,-0.709005,-0.748531,-0.702509,-0.707116,-0.738694,-0.345208,-0.648736,-0.974071,-0.797944,0.000000
50%,72.060419,3.843531,28.061088,225.274427,41.315515,116.678685,100.401232,55.787927,1474.387134,2.643519,...,-0.015920,-0.117113,0.029917,-0.045186,0.008325,0.295052,-0.050378,-0.032262,-0.062211,0.000000
75%,79.482150,4.784608,39.393170,235.982436,49.664245,167.173128,130.178333,63.468424,1660.918793,3.824170,...,0.658691,0.544126,0.814158,0.606528,0.576091,0.848949,0.568884,0.512326,0.591464,1.000000
max,101.530691,6.454887,61.162743,252.134333,71.790180,292.716198,223.011404,85.989007,2184.070195,6.871534,...,2.664860,2.226981,2.587329,2.165415,2.875425,2.266812,2.495302,3.072234,2.619726,1.000000


# **2. Example Solution: Logistic Regression (Baseline)**

This baseline model uses **only the labeled training data**, and fits a simple logistic regression on all the numeric features.

In [8]:
# Preparation: we convert the Gyártósor column to numeric
le = LabelEncoder()
all_lines = pd.concat([train_labeled['Gyártósor'], train_unlabeled['Gyártósor'], test_df['Gyártósor']])
le.fit(all_lines)

# Features and target variable from the labeled data
X_train = train_labeled.drop(columns=['ID', 'Selejt']).copy()
X_train['Gyártósor'] = le.transform(X_train['Gyártósor'])
y_train = train_labeled['Selejt']

# Test data
X_test = test_df.drop(columns=['ID']).copy()
X_test['Gyártósor'] = le.transform(X_test['Gyártósor'])

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## **2.1 Training the Model and Making Predictions**

In [9]:
# Training the logistic regression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Prediction on the test data (probability values)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print(f"Mean of the predictions: {y_pred_proba.mean():.4f}")
print(f"Predictions min: {y_pred_proba.min():.4f}, max: {y_pred_proba.max():.4f}")

Mean of the predictions: 0.5040
Predictions min: 0.0044, max: 0.9897


## **2.2 Saving the Predictions**

In [10]:
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'Selejt': y_pred_proba
})

# Don't forget to include your identifier and the submission number in the file name!
submission.to_csv('HUNXYZ_A_X.csv', index=False)
print(f"Submission file saved: {len(submission)} rows")
submission.head()

Submission file saved: 2000 rows


,ID,Selejt
0,ITEM_13000,0.393345
1,ITEM_13001,0.913331
2,ITEM_13002,0.847496
3,ITEM_13003,0.480969
4,ITEM_13004,0.454425


# **3. Uploading Predictions: Submission Guide**

### **Example CSV Format**

```csv
ID,Selejt
ITEM_10000,0.405314
ITEM_10001,0.401229
ITEM_10002,0.401746
...
```

### **File naming convention**

- The file name should be: *\<received_identifier>\_\<task>\_\<checkpoint_number>.csv*
  - `<received_identifier>` is the unique identifier you received (e.g. `HUN123`)
  - `<task>` is the label of the given task: `A` or `B`
  - `<checkpoint_number>` is the sequence number of the given submission (e.g. `1`, `2`, etc.)

**Example**: `HUN123_A_1.csv`

- Name the notebook file belonging to the prediction **according to the same convention** (`.ipynb`):

**Example**: `HUN123_A_1.ipynb`

### **Finalizing your submission**

- Upload the `.csv` prediction file to **Kaggle**, and submit the corresponding `.ipynb` notebook file through the **upload interface**: [https://tehetseg.inf.elte.hu/mi_olimpia/dock/kaggle?comp=magyar-mi-diakolimpia-online-valogato-2026-b](https://tehetseg.inf.elte.hu/mi_olimpia/dock/kaggle?comp=magyar-mi-diakolimpia-online-valogato-2026-b)
- **When registering on Kaggle, use the same e-mail address that you provided when registering for the competition.**
- **On the Kaggle platform, your username should be your own name**: we cannot accept submissions made under pseudonyms or aliases.
- You must submit the notebook belonging to your selected submission through the upload interface **within 15 minutes after the competition closes**. If it is unavailable, you may also send it to the **midiakolimpia@gmail.com** e-mail address.
- Verification and the calculation of the private score are based on the selected submission and the submitted notebook.

### **Multiple submissions**

- In case of multiple attempts, use increasing checkpoint numbers (e.g. `HUN123_A_2.csv`, `HUN123_A_3.csv`, etc.)
- Upload the `.csv` prediction file to Kaggle, and submit the corresponding `.ipynb` notebook file on the website, with a matching name.
- If you solve both tasks, we ask for a separate notebook for each (e.g. `HUN123_A_1.ipynb` and `HUN123_B_1.ipynb`).

---

**Don't forget:** the notebook and the prediction file are valid *as a pair*, so always pay attention to the exact file naming and saving!